# 04 Modelo Regresion Total Pedido

Este notebook usa la base `EDA`, entrena una `Regresion Lineal`, estima el total del pedido y exporta el `Parquet` final de la etapa `04`.


## 1. Librerias


In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


## 2. Definir rutas


In [ ]:
from pathlib import Path

def resolve_project_root() -> Path:
    # Buscar la raiz del proyecto a partir del directorio actual.
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "parquets").exists():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raiz del proyecto")

PROJECT_ROOT = resolve_project_root()

# Definir las rutas de entrada y salida de la etapa 04.
INPUT_PATH = PROJECT_ROOT / "parquets" / "02_EDA_Base_Tickets" / "02_base_eda_tickets.parquet"
OUTPUT_DIR = PROJECT_ROOT / "parquets" / "04_Modelo_Regresion_Total_Pedido"
OUTPUT_PATH = OUTPUT_DIR / "04_tickets_regresion.parquet"
MODEL_DIR = PROJECT_ROOT / "models" / "04_Modelo_Regresion_Total_Pedido"
MODEL_PATH = MODEL_DIR / "04_modelo_regresion_lineal.joblib"
FEATURES_PATH = MODEL_DIR / "04_features_regresion_lineal.json"

print(f"Raiz del proyecto: {PROJECT_ROOT}")
print(f"Parquet de entrada: {INPUT_PATH}")
print(f"Parquet de salida: {OUTPUT_PATH}")
print(f"Modelo entrenado: {MODEL_PATH}")


## 3. Cargar la base EDA


In [ ]:
# Leer la base EDA de entrada.
df = pd.read_parquet(INPUT_PATH)

# Mostrar las primeras filas de la base para regresion.
df.head()


### Resultado breve

La base de entrada de esta etapa permite estimar un monto real del negocio, en este caso el `total_pedido` de cada ticket.


## 4. Seleccionar variables y target


In [ ]:
# Definir las variables explicativas del modelo.
FEATURES = [
    "dia", "mes", "trimestre", "dia_semana", "dia_tipo", "fin_semana",
    "ciudad", "capacidad_sucursal", "tipo_empleado", "salario", "turno",
    "numero_mesa", "capacidad_mesa", "metodo_pago", "lineas_ticket",
    "cantidad_total", "platillos_distintos", "categorias_distintas",
    "incluye_bebida", "incluye_postre", "incluye_entrada", "incluye_platillo_fuerte"
]

# Definir la variable objetivo.
TARGET = "total_pedido"

# Separar X y y para el entrenamiento.
X = df[FEATURES].copy()
y = df[TARGET].copy()
X.head()


## 5. Particion de entrenamiento y prueba


In [ ]:
# Dividir la base en entrenamiento y prueba.
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.2, random_state=42
)

# Revisar dimensiones de las particiones.
X_train.shape, X_test.shape


## 6. Entrenar y evaluar el modelo


In [ ]:
# Separar columnas numericas y categoricas.
numericas = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categoricas = X.select_dtypes(include=["object"]).columns.tolist()

# Definir el preprocesador del pipeline.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numericas,
        ),
        (
            "cat",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categoricas,
        ),
    ]
)

# Definir el pipeline completo de regresion.
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression()),
])

# Entrenar el modelo con la muestra de entrenamiento.
pipeline.fit(X_train, y_train)

# Generar predicciones sobre test.
y_pred = pipeline.predict(X_test)

# Resumir metricas principales.
metricas = pd.DataFrame(
    {
        "metrica": ["mae", "rmse", "r2"],
        "valor": [
            round(float(mean_absolute_error(y_test, y_pred)), 4),
            round(float(np.sqrt(mean_squared_error(y_test, y_pred))), 4),
            round(float(r2_score(y_test, y_pred)), 4),
        ],
    }
)
metricas


### Resultado breve

Las metricas de error permiten revisar que tan cerca queda el modelo del monto real del pedido y sirven como linea base para futuras mejoras.


## 7. Visualizaciones del modelo


In [ ]:
# Construir una vista rapida de resultados sobre test.
resultados_test = X_test.copy()
resultados_test["total_real"] = y_test
resultados_test["total_pred"] = y_pred
resultados_test["residuo"] = resultados_test["total_real"] - resultados_test["total_pred"]

# Graficar comparacion entre total real y total predicho.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=resultados_test, x="total_real", y="total_pred", ax=axes[0])
min_val = min(resultados_test["total_real"].min(), resultados_test["total_pred"].min())
max_val = max(resultados_test["total_real"].max(), resultados_test["total_pred"].max())
axes[0].plot([min_val, max_val], [min_val, max_val], "r--")
axes[0].set_title("Total real vs total predicho")
axes[0].set_xlabel("total_real")
axes[0].set_ylabel("total_pred")

# Graficar la distribucion del residuo.
sns.histplot(resultados_test["residuo"], bins=30, kde=True, ax=axes[1])
axes[1].set_title("Distribucion del residuo")
axes[1].set_xlabel("residuo")
plt.tight_layout()


### Resultado breve

Estas graficas muestran tanto el ajuste general del modelo como el comportamiento del error en la muestra de prueba.


## 8. Exportar el parquet final del modelo


In [ ]:
# Crear carpetas de salida para la etapa 04.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Entrenar el modelo final con toda la base disponible.
final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression()),
])
final_pipeline.fit(X, y)

# Generar la salida final con predicciones y errores por ticket.
df_predictions = df.copy()
df_predictions["pred_total_pedido"] = final_pipeline.predict(X)
df_predictions["residuo_total_pedido"] = df_predictions["total_pedido"] - df_predictions["pred_total_pedido"]
df_predictions["error_abs_total_pedido"] = df_predictions["residuo_total_pedido"].abs()
df_predictions["modelo"] = "Regresion Lineal"
df_predictions["fuente_parquet"] = "02_base_eda_tickets.parquet"
df_predictions["conjunto_evaluacion"] = "train"
df_predictions.loc[idx_test, "conjunto_evaluacion"] = "test"

test_lookup = pd.DataFrame(
    {
        "idx": idx_test,
        "pred_total_pedido_test": y_pred,
        "residuo_total_pedido_test": y_test.to_numpy() - y_pred,
    }
).set_index("idx")

df_predictions = df_predictions.join(test_lookup, how="left")

# Exportar el parquet final y los artefactos del modelo.
df_predictions.to_parquet(OUTPUT_PATH, index=False)
joblib.dump(final_pipeline, MODEL_PATH)
FEATURES_PATH.write_text(json.dumps(FEATURES, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Parquet generado en: {OUTPUT_PATH}")
print(f"Modelo guardado en: {MODEL_PATH}")


In [ ]:
# Verificar la salida exportada de la etapa 04.
df_exportado = pd.read_parquet(OUTPUT_PATH)
df_exportado.shape


### Resultado breve

La etapa `04` queda cerrada cuando el parquet de regresion ya incluye el valor estimado, el residuo y el error absoluto por ticket.
